In [ ]:
import sys
sys.executable

In [ ]:
from pathlib import Path
import pandas as pd

CAMINHO = Path("../data/raw/arquivo-final-3-trim-26.csv")

# Limites aproximados do município de São Paulo, definidos UMA vez e usados no notebook todo.
LAT_MIN, LAT_MAX = -24.01, -23.35
LON_MIN, LON_MAX = -46.83, -46.36

df = pd.read_csv(
    CAMINHO,
    sep=",",
    encoding="utf-8",
    dtype=str,   # tudo como texto: nada é convertido ou perdido na leitura
)

In [ ]:
df.shape
df.info()

In [ ]:
nulos = df.isna().sum()
pd.DataFrame({"nulos": nulos, "%": (nulos / len(df) * 100).round(1)})

In [ ]:
df.duplicated().sum()

In [ ]:
abertura = pd.to_datetime(df["Data de Abertura"], format="%m/%d/%Y", errors="coerce")
finalizacao = pd.to_datetime(df["Data de Finalização"], format="%m/%d/%Y", errors="coerce")

print("Abertura:", abertura.min(), "->", abertura.max())
print("Falhas na conversão (abertura):", abertura.isna().sum(), "| nulos originais:", df["Data de Abertura"].isna().sum())
print("Falhas na conversão (finalização):", finalizacao.isna().sum(), "| nulos originais:", df["Data de Finalização"].isna().sum())

In [ ]:
display(df["Status"].value_counts(dropna=False))
display(df["Meio de Comunicação"].value_counts(dropna=False))

In [ ]:
df["Status"].value_counts(dropna=False)

In [ ]:
# percentual de nulos por coluna (vale também para string vazia)
df = df.replace("", pd.NA)
(df.isna().mean() * 100).round(1).sort_values(ascending=False)

In [ ]:
# serviços que contêm as palavras-chave, com contagem
mask = df["Serviço"].str.contains("buraco|paviment|ilumina", case=False, na=False)
df.loc[mask, ["Tema", "Assunto", "Serviço"]].value_counts()

In [ ]:
# Colunas numéricas novas; as originais em texto ficam intactas para comparação.
df["lat"] = pd.to_numeric(df["Latitude"], errors="coerce")
df["lon"] = pd.to_numeric(df["Longitude"], errors="coerce")

# Conferência: texto preenchido que NÃO virou número indica formato inesperado.
falhas = df["Latitude"].notna() & df["lat"].isna()
print("Latitudes em texto que falharam na conversão:", falhas.sum())
df[["Latitude", "lat", "Longitude", "lon"]].dropna().head(5)

In [ ]:
dentro_sp = df["lat"].between(LAT_MIN, LAT_MAX) & df["lon"].between(LON_MIN, LON_MAX)
sem_coord = df["lat"].isna() | df["lon"].isna()
zero      = (df["lat"] == 0) | (df["lon"] == 0)
fora_sp   = ~sem_coord & ~zero & ~dentro_sp   # tem coordenada, não é zero, mas cai fora

print(f"Dentro de SP:            {dentro_sp.sum():>8,}")
print(f"Sem coordenada:          {sem_coord.sum():>8,}")
print(f"Coordenada 0,0:          {zero.sum():>8,}")
print(f"Fora de SP (geocodif.):  {fora_sp.sum():>8,}")
print(f"Total:                   {len(df):>8,}")

In [ ]:
df[["lat", "lon"]].describe()

In [ ]:
(finalizacao < abertura).sum()

In [ ]:
# Agrupa as colunas, conta as repetições e cria uma nova coluna chamada 'Quantidade'
df_tema = df.value_counts(["Tema"]).reset_index(name="Quantidade")

# Mostra as primeiras linhas do resultado
print(df_tema.head())

In [ ]:
# Agrupa as colunas, conta as repetições e cria uma nova coluna chamada 'Quantidade'
df_servico = df.value_counts(["Serviço"]).reset_index(name="Quantidade")

# Mostra as primeiras linhas do resultado
print(df_servico.head())

In [ ]:
# Agrupa as colunas, conta as repetições e cria uma nova coluna chamada 'Quantidade'
df_assunto = df.value_counts(["Assunto"]).reset_index(name="Quantidade")

# Mostra as primeiras linhas do resultado
print(df_assunto.head())

In [ ]:
# Cria a tabela cruzada
tabela_status = pd.crosstab(
    index=df["Status"], 
    columns=df["Data de Finalização"].isna(),
    rownames=["Status"],
    colnames=["Data Nula?"]
)

# Exibe o resultado
print(tabela_status)

In [ ]:
# Filtra o DataFrame apenas para o tema desejado
df_rua_bairro = df[df["Tema"] == "Rua e bairro"]

top_15_servicos = df_rua_bairro["Serviço"].value_counts().head(15)

print(top_15_servicos)

In [ ]:
padrao = "ilumina|lâmpada|lampada|poste|luz|apagad"
filtro_iluminacao = (
    df["Tema"].str.contains(padrao, case=False, na=False)
    | df["Assunto"].str.contains(padrao, case=False, na=False)
    | df["Serviço"].str.contains(padrao, case=False, na=False)
)
# O total sozinho engana: é preciso ver QUAIS serviços a busca pegou.
df.loc[filtro_iluminacao, ["Assunto", "Serviço"]].value_counts()

In [ ]:
df_tapa = df[df["Serviço"] == "Tapa-buraco"].copy()
valido_tapa = df_tapa["lat"].between(LAT_MIN, LAT_MAX) & df_tapa["lon"].between(LON_MIN, LON_MAX)

qtd_invalidos = (~valido_tapa).sum()
print(f"Tapa-buraco: {len(df_tapa):,}")
print(f"Sem coordenada válida (nulo, zero ou fora de SP): {qtd_invalidos} "
      f"({qtd_invalidos / len(df_tapa):.2%})")

df_tapa_valido = df_tapa[valido_tapa].copy()

In [ ]:
criterios = {
    "mesma coordenada, qualquer data": ["lat", "lon"],
    "mesma coordenada e mesmo dia":    ["lat", "lon", "Data de Abertura"],
}
total = len(df_tapa_valido)
for nome, colunas in criterios.items():
    qtd = df_tapa_valido.duplicated(subset=colunas, keep="first").sum()
    print(f"{nome:<35} {qtd:>6,} ({qtd / total:.1%})")

# Quantos chamados por ponto? Mostra se a repetição é rara ou concentrada em poucos locais.
por_ponto = df_tapa_valido.groupby(["lat", "lon"]).size()
por_ponto.value_counts().sort_index().head(10)

In [ ]:
top = por_ponto.nlargest(10)
print(top)
# Para o ponto com mais chamados: os endereços são os mesmos ou diferentes?
lat0, lon0 = top.index[0]
df_tapa_valido.query("lat == @lat0 and lon == @lon0")[["Logradouro", "Número", "Data de Abertura"]]

In [ ]:
# Todo o trecho: os 3 pontos vizinhos juntos (caixa de ~150 m)
trecho = df_tapa_valido[
    df_tapa_valido["lat"].between(-23.4610, -23.4590)
    & df_tapa_valido["lon"].between(-46.5725, -46.5705)
]
print(len(trecho), "chamados no trecho")

# 1) Quais endereços e números aparecem no trecho?
display(trecho[["Logradouro", "Número"]].value_counts())

# 2) Por qual canal chegaram?
display(trecho["Meio de Comunicação"].value_counts())

# 3) Como os chamados se distribuem no tempo e que status têm?
display(pd.crosstab(trecho["Data de Abertura"], trecho["Status"]))

# 4) O número "2" é comum na base toda? 
display(df_tapa_valido["Número"].value_counts().head(10))